# Testing the Baseline RAG System


In [1]:
import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import faithfulness,answer_relevancy,context_recall,context_precision
import requests
from dotenv import load_dotenv
import os 

In [2]:
os.environ["LANGSMITH_PROJECT"]="RAGAS"

In [3]:
import asyncio,aiohttp

In [4]:
load_dotenv()

True

In [5]:

with open('../data/eval_dataset.json') as f:
    raw = json.load(f)

In [6]:
records = raw['ragas_dataset']

In [11]:
records

[{'id': 1,
  'category': 'exact_keyword',
  'question': 'What is the ClinicalTrials.gov registration number for the CLARITY-1 trial?',
  'ground_truth': 'The CLARITY-1 trial is registered under NCT-2021-4412.',
  'contexts': ['MBS-101 (velaritinib): Velaritinib is a selective, orally bioavailable inhibitor of epidermal growth factor receptor (EGFR) mutant variants, specifically targeting the L858R and exon 19 deletion mutations that account for approximately 85% of sensitising EGFR mutations in NSCLC patients. The NDA filing on November 30, 2024 is supported by data from the CLARITY-1 Phase III trial and two supportive Phase II studies (NCT-2019-1143 and NCT-2020-2287).',
   'CLARITY-1 (NCT-2021-4412) was a randomised, double-blind, placebo-controlled, multicentre Phase III trial evaluating velaritinib versus placebo as a first-line monotherapy in patients with locally advanced or metastatic NSCLC harbouring sensitising EGFR mutations.']},
 {'id': 2,
  'category': 'exact_keyword',
  'q

In [7]:
eval_rows = []

In [8]:
CONCURRENCY_LIMIT = 1

In [9]:
# single request function
async def fetch_answer(session,semaphore, question):
    async with semaphore:
        async with session.post(
            "http://localhost:8000/retrieve",
            json={"query": question}
        ) as resp:
            return await resp.json()


In [10]:
# batch processing function
async def run_all():
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_answer(session, semaphore, r["question"]) for r in records]
        return await asyncio.gather(*tasks)

In [12]:
responses = await run_all()

In [13]:
for r, response in zip(records, responses):
    eval_rows.append({
        "question": r["question"],
        "answer": response["answer"],
        "context": [s['document'] for s in response['sources']],
        "ground_truth": r['ground_truth']
    })

In [14]:
dataset = Dataset.from_dict({
    "question": [r["question"]          for r in eval_rows],
    "answer": [r["answer"]             for r in eval_rows],
    "retrieved_contexts": [r['context'] for r in eval_rows],
    "ground_truth": [r['ground_truth'] for r in eval_rows]
})

In [ ]:
dataset.save_to_disk('../data/ragas_eval_dataset')
with open('../data/ragas_eval_results_baseline_rag.json', 'w') as f:
    json.dump(eval_rows, f, indent=2)
print(f'Saved {len(dataset)} rows to data/ragas_eval_dataset/ and data/ragas_eval_results.json')

Saving the dataset (0/1 shards):   0%|          | 0/25 [00:00<?, ? examples/s]

Saved 25 rows to data/ragas_eval_dataset/ and data/ragas_eval_results.json


In [16]:
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecision
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from langchain_mistralai import ChatMistralAI,MistralAIEmbeddings

mistral_llm = LangchainLLMWrapper(ChatMistralAI(model="mistral-small-2506", temperature=0))
mistral_embedder = MistralAIEmbeddings(model="mistral-embed-2312")

faithfulness_metric = Faithfulness(llm=mistral_llm)
answer_relevancy_metric = AnswerRelevancy(llm=mistral_llm,embeddings = mistral_embedder)
context_recall_metric = ContextRecall(llm=mistral_llm)
context_precision_metric = ContextPrecision(llm=mistral_llm)


C:\Users\ankit\AppData\Local\Temp\ipykernel_11608\453571439.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecision
C:\Users\ankit\AppData\Local\Temp\ipykernel_11608\453571439.py:1: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecision
C:\Users\ankit\AppData\Local\Temp\ipykernel_11608\453571439.py:1: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.

In [17]:
result = evaluate(dataset, metrics=[faithfulness_metric, answer_relevancy_metric, context_precision_metric, context_recall_metric],
                  llm=mistral_llm,
                  embeddings=mistral_embedder,
                  run_config=RunConfig(
                                        max_workers=10,
                                        timeout=180
                                        ))

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
import numpy as np
result

{'faithfulness': 0.4348, 'answer_relevancy': 0.6797, 'context_precision': 0.6500, 'context_recall': 0.7467}

In [50]:
from langsmith import Client

client = Client()

# Assuming you have the run_id for each interaction
# and a `result` dictionary from Ragas
run_id = "019e6d32-a41c-7650-bd56-26e0093af235"

# Log Faithfulness
client.create_feedback(
    run_id=run_id,
    key="faithfulness",
    score=round(np.mean(result["faithfulness"]),2),
)

# Log Context Precision
client.create_feedback(
    run_id=run_id,
    key="context_precision",
    score=round(np.mean(result["context_precision"]),2),
)
# Log Context Recall
client.create_feedback(
    run_id=run_id,
    key="context_recall",
    score=round(np.mean(result["context_recall"]),2),
)
# Log Answer Relevancy
client.create_feedback(
    run_id=run_id,
    key="answer_relevancy",
    score=round(np.mean(result["answer_relevancy"]),2),
)


Feedback(id=UUID('019e6d97-35c4-7d31-9e80-dac05220f85e'), created_at=datetime.datetime(2026, 5, 28, 7, 58, 6, 788383, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 5, 28, 7, 58, 6, 788387, tzinfo=datetime.timezone.utc), run_id=UUID('019e6d32-a41c-7650-bd56-26e0093af235'), trace_id=None, key='answer_relevancy', score=0.68, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='api', metadata={}, user_id=None, user_name=None), session_id=None, start_time=None, comparative_experiment_id=None, feedback_group_id=None, extra=None)

### --------------------------------------- END OF EVALUATION ----------------------------------------------------------

# CRAG EVALUATION